### Failure-Day Validation

Checked whether ornot the training dataset has observations where `failure == 1`.

No failure-day rows were found which confims that the observations where the drive had already failed were excluded from the training data. 

In [ ]:
import pandas as pd

training_df = pd.read_csv("../data/processed/training_data.csv")

print(training_df[training_df['failure'] == 1])


Empty DataFrame
Columns: [serial_number, date, model, capacity_bytes, failure, datacenter, cluster_id, vault_id, pod_id, pod_slot_num, is_legacy_format, smart_1_normalized, smart_1_raw, smart_2_normalized, smart_2_raw, smart_3_normalized, smart_3_raw, smart_4_normalized, smart_4_raw, smart_5_normalized, smart_5_raw, smart_7_normalized, smart_7_raw, smart_8_normalized, smart_8_raw, smart_9_normalized, smart_9_raw, smart_10_normalized, smart_10_raw, smart_11_normalized, smart_11_raw, smart_12_normalized, smart_12_raw, smart_13_normalized, smart_13_raw, smart_15_normalized, smart_15_raw, smart_16_normalized, smart_16_raw, smart_17_normalized, smart_17_raw, smart_18_normalized, smart_18_raw, smart_22_normalized, smart_22_raw, smart_23_normalized, smart_23_raw, smart_24_normalized, smart_24_raw, smart_27_normalized, smart_27_raw, smart_71_normalized, smart_71_raw, smart_82_normalized, smart_82_raw, smart_90_normalized, smart_90_raw, smart_160_normalized, smart_160_raw, smart_161_normalized,

### Target Value Validation

Check that `failure_within_30_days` contains only the expected labels `0` and `1`.

In [20]:
print(training_df['failure_within_30_days'].unique())
print(training_df['failure_within_30_days'].isna().sum())

[1 0]
0


### Duplicate Observation Validation

Check whether or not the same drive appears more than once on the same date.

Each `serial_number` should have only one observation per day, so duplicate `serial_number` and `date` combinations could cause the same observation to be counted multiple times during training.

In [25]:
print(training_df.duplicated(subset=['serial_number', 'date']).sum())

0


### Manual Failed-Drive History Validation

Looked into 3 drives that eventually failed to verify that their labels transition correctly as the failure date approaches.

Observations more than 30 days before failure should have `failure_within_30_days = 0`. Once a drive enters the 30-day period before its failure, the label should change to `failure_within_30_days = 1` and remain positive as the failure date approaches.

In [ ]:
drive_history1 = training_df[(training_df['serial_number'] == 'ZL2CV1VC') & (training_df['date'] >= '2026-02-25') & (training_df['date'] <= '2026-03-05')]

print(drive_history1[['serial_number', 'date', 'failure_within_30_days']].head(70))

      serial_number        date  failure_within_30_days
42091      ZL2CV1VC  2026-02-25                       0
42092      ZL2CV1VC  2026-02-26                       0
42093      ZL2CV1VC  2026-02-27                       0
42094      ZL2CV1VC  2026-02-28                       0
42095      ZL2CV1VC  2026-03-01                       1
42096      ZL2CV1VC  2026-03-02                       1
42097      ZL2CV1VC  2026-03-03                       1
42098      ZL2CV1VC  2026-03-04                       1
42099      ZL2CV1VC  2026-03-05                       1


In [ ]:
drive_history2 = training_df[(training_df['serial_number'] == 'ZL23X50P') & (training_df["date"] >= "2026-02-25") & (training_df["date"] <= "2026-03-05")]

print(drive_history2[['serial_number', 'date', 'failure_within_30_days']].head(70))

      serial_number        date  failure_within_30_days
42002      ZL23X50P  2026-02-25                       0
42003      ZL23X50P  2026-02-26                       0
42004      ZL23X50P  2026-02-27                       0
42005      ZL23X50P  2026-02-28                       0
42006      ZL23X50P  2026-03-01                       1
42007      ZL23X50P  2026-03-02                       1
42008      ZL23X50P  2026-03-03                       1
42009      ZL23X50P  2026-03-04                       1
42010      ZL23X50P  2026-03-05                       1


In [78]:
drive_history3 = training_df[(training_df['serial_number'] == '10G0A004F97G') & (training_df["date"] >= "2026-02-10") & (training_df["date"] <= "2026-02-20")]

print(drive_history3[['serial_number', 'date', 'failure_within_30_days']].head(70))

      serial_number        date  failure_within_30_days
32372  10G0A004F97G  2026-02-10                       0
32373  10G0A004F97G  2026-02-11                       0
32374  10G0A004F97G  2026-02-12                       0
32375  10G0A004F97G  2026-02-13                       0
32376  10G0A004F97G  2026-02-14                       0
32377  10G0A004F97G  2026-02-15                       0
32378  10G0A004F97G  2026-02-16                       0
32379  10G0A004F97G  2026-02-17                       1
32380  10G0A004F97G  2026-02-18                       1
32381  10G0A004F97G  2026-02-19                       1
32382  10G0A004F97G  2026-02-20                       1


### Manual Healthy-Drive History Validation

Inspect drives labeled as negative examples to verify that their observations consistently have `failure_within_30_days = 0`.

In [79]:
healthy_drives = training_df.groupby("serial_number")["failure_within_30_days"].max()

print(healthy_drives[healthy_drives == 0].head(10))

serial_number
0702152b2e210010    0
1040A012F97G        0
1050A06WF97G        0
1050A076F97G        0
1050A07UF97G        0
1050A0BGF97G        0
1050A0BHF97G        0
1060A019F97G        0
1060A01AF97G        0
1060A046F9RG        0
Name: failure_within_30_days, dtype: int64


In [80]:
healthy_history1 = training_df[training_df["serial_number"] == "0702152b2e210010"]

print(healthy_history1[["serial_number", "date", "failure_within_30_days"]])

          serial_number        date  failure_within_30_days
51299  0702152b2e210010  2026-01-10                       0
52140  0702152b2e210010  2026-01-11                       0
53140  0702152b2e210010  2026-01-12                       0
54142  0702152b2e210010  2026-01-13                       0
55167  0702152b2e210010  2026-01-14                       0
56186  0702152b2e210010  2026-01-15                       0
57166  0702152b2e210010  2026-01-16                       0
58295  0702152b2e210010  2026-01-17                       0
59158  0702152b2e210010  2026-01-18                       0
60172  0702152b2e210010  2026-01-19                       0
61148  0702152b2e210010  2026-01-20                       0
62366  0702152b2e210010  2026-01-21                       0
63354  0702152b2e210010  2026-01-22                       0
65691  0702152b2e210010  2026-01-24                       0
66236  0702152b2e210010  2026-01-25                       0
71823  0702152b2e210010  2026-01-30     

In [84]:
healthy_history2 = training_df[training_df["serial_number"] == "1060A046F9RG"]

print(healthy_history2[["serial_number", "date", "failure_within_30_days"]])

       serial_number        date  failure_within_30_days
99990   1060A046F9RG  2026-02-27                       0
100861  1060A046F9RG  2026-02-28                       0
101873  1060A046F9RG  2026-03-01                       0


### Data Leakage Review

Review the columns in the training dataset to identify information that should not be used as input to the machine learning model. The features that reveal the outcome directly or do not represent useful drive health information should be excluded before training.

In [85]:
print(training_df.columns.tolist())

['serial_number', 'date', 'model', 'capacity_bytes', 'failure', 'datacenter', 'cluster_id', 'vault_id', 'pod_id', 'pod_slot_num', 'is_legacy_format', 'smart_1_normalized', 'smart_1_raw', 'smart_2_normalized', 'smart_2_raw', 'smart_3_normalized', 'smart_3_raw', 'smart_4_normalized', 'smart_4_raw', 'smart_5_normalized', 'smart_5_raw', 'smart_7_normalized', 'smart_7_raw', 'smart_8_normalized', 'smart_8_raw', 'smart_9_normalized', 'smart_9_raw', 'smart_10_normalized', 'smart_10_raw', 'smart_11_normalized', 'smart_11_raw', 'smart_12_normalized', 'smart_12_raw', 'smart_13_normalized', 'smart_13_raw', 'smart_15_normalized', 'smart_15_raw', 'smart_16_normalized', 'smart_16_raw', 'smart_17_normalized', 'smart_17_raw', 'smart_18_normalized', 'smart_18_raw', 'smart_22_normalized', 'smart_22_raw', 'smart_23_normalized', 'smart_23_raw', 'smart_24_normalized', 'smart_24_raw', 'smart_27_normalized', 'smart_27_raw', 'smart_71_normalized', 'smart_71_raw', 'smart_82_normalized', 'smart_82_raw', 'smart_9

- `failure` should not be used as a model feature because it represents if the drive has already failed on that day.
- `serial_number` is an identifier and not a health measurement.
- `failure_within_30_days` is the target that the model will predict, so it must not be included in the input features.